# Ноутбук 2 — гружу модель и проверяю, что она работает

Тут беру базовую модель `deepvk/llava-gemma-2b-lora` и загружаю её в 4-bit, чтобы влезла в T4.
Дальше проверяю инференс: даю картинку и вопрос, смотрю, что ответит. И в конце смотрю,
сколько заняло видеопамяти — прикидываю, останется ли место под дообучение.

## Ставлю библиотеки

Pillow беру версии ниже 12 — с новой в Colab ломается импорт transformers.

In [ ]:
!pip install -q -U datasets huggingface_hub transformers peft accelerate bitsandbytes evaluate
!pip install -q -U "pillow<12"

## Логинюсь в HuggingFace

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Гружу модель в 4 бита

Процессору руками проставляю patch_size и пару настроек — в старом конфиге модели их нет,
а новый transformers без них падает на инференсе.

In [ ]:
import torch
from transformers import LlavaForConditionalGeneration, AutoProcessor, AutoTokenizer, BitsAndBytesConfig

model_id = 'deepvk/llava-gemma-2b-lora'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb,
    torch_dtype=torch.float16,
    device_map='auto',
)
processor = AutoProcessor.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

processor.patch_size = 14
processor.vision_feature_select_strategy = 'default'
processor.num_additional_image_tokens = 1

print('модель загрузилась')
print(model.config.model_type)

## Пробую инференс на картинке из примера

Беру ту же картинку со стоп-знаком, что в карточке модели, и прошу описать её.

In [ ]:
import requests
from PIL import Image

url = 'https://www.ilankelman.org/stopsigns/australia.jpg'
img = Image.open(requests.get(url, stream=True).raw).convert('RGB')
img

In [ ]:
messages = [
    {'role': 'user', 'content': '<image>\nОпиши картинку несколькими словами.'}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(images=[img], text=text, return_tensors='pt').to(model.device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=30)

answer = tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(answer)

## Смотрю, сколько заняло видеопамяти

In [ ]:
used = torch.cuda.memory_allocated() / 1e9
peak = torch.cuda.max_memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'сейчас занято : {used:.2f} GB')
print(f'пик          : {peak:.2f} GB')
print(f'всего на GPU : {total:.2f} GB')
print(f'свободно     : {total - peak:.2f} GB')

## Что получилось

Если модель загрузилась и на картинку выдала осмысленный ответ — базовая часть работает,
инференс в 4-bit на T4 идёт. По памяти видно, сколько осталось свободно под дообучение
(LoRA добавит немного сверху — адаптеры + оптимизатор).

Дальше: прогнать эту модель без обучения на GQA-ru и MMBench-ru и записать метрики (точка отсчёта).